# Camargo Domestic Declarations — SharedCat architecture + role-grouped pickles + n-gram loader

DomDecl profile: ~10.5k cases, ~56k events, average trace ~5–6 events, simple SF, steady TV. Closer to helpdesk than to BPIC17 — same architecture / training-loop choices apply.

Three changes relative to the original `train_camargo_LSTM.ipynb`:

1. **Role-grouped pickles** (`domestic_declarations_all_5_roles_*.pkl`) — `model_feat = [['Activity', 'Role'], ['case_elapsed_time']]`. DomDecl's XES already ships a role hierarchy (Employee, Supervisor, Director, Pre-approver, Budget Owner, Missing, Administration); we use it directly instead of clustering raw Resource values. Steady TV ⇒ max-scaled time, not log.
2. **SharedCat architecture** — `sharedCatLSTM.model.SharedCat_LSTM` (paper Fig 6b, simple SF / steady TV row).
3. **N-gram loader (Camargo §3.1 / Table 1)** — `training.camargo_ngram_dataset.CamargoNGramDataset` with `NGRAM_SIZE = 5`. Replaces the U-ED-LSTM-style encoder-decoder windows.

Also fixes the stale carryover save path (`Sepsis_camargo_act_1_suffix_length2.pkl`) → `DomesticDeclarations_camargo_sharedcat_role_ngram5.pkl`.

Reimplementation for comparison:
- Camargo, Manuel, Marlon Dumas, and Oscar González-Rojas. "Learning accurate LSTM models of business processes." BPM 2019.
- https://github.com/AdaptiveBProcess/GenerativeLSTM/

# Imports

In [9]:
import importlib
import sys
import torch
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir() and (_current / 'data').is_dir():
        break
    _current = _current.parent
_project_root = _current

for p in [
    _project_root,
    _project_root / 'src',
    _project_root / 'src' / 'reimplemented_comparable_approaches' / 'camargo_LSTM_suffix_pred',
]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Data

In [10]:
file_path_train = '../../Loader/pkl/domestic_declarations_all_5_roles_train.pkl'
dd_train_dataset = torch.load(file_path_train, weights_only=False)
print(type(dd_train_dataset))

file_path_val = '../../Loader/pkl/domestic_declarations_all_5_roles_val.pkl'
dd_val_dataset = torch.load(file_path_val, weights_only=False)
print(type(dd_val_dataset))


<class 'event_log_loader.new_event_log_loader.EventLogDataset'>
<class 'event_log_loader.new_event_log_loader.EventLogDataset'>


In [11]:
# SKIPPED — this cell used to overwrite the role-grouped dataset with the
# non-roles pickles, which conflicts with `camargo_test_pickle` (Roles) in the
# config. Kept here for historical reference; do not re-enable for SharedCat.


### Train Data Insights

In [12]:
# DomesticDeclarations Dataset Categories, Features:
dd_all_categories = dd_train_dataset.all_categories

dd_all_categories_cat = dd_all_categories[0]
print(dd_all_categories_cat)

dd_all_categories_num = dd_all_categories[1]
print(dd_all_categories_num)

for i, cat in enumerate(dd_all_categories_cat):
     print(f"domestic_declarations(5) Categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"domestic_declarations (5) Total Amount of Category labels: {cat[1]}")

print('\n')    

for i, num in enumerate(dd_all_categories_num):
     print(f"domestic_declarations (5) Numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"domestic_declarations (5) Amount Numerical: {num[1]}")
     
# Get concept_name id:
concept_name = 'Activity'
concept_name_id = [i for i, cat in enumerate(dd_all_categories[0]) if cat[0] == concept_name][0]
print("ID concet name in cat list: ", concept_name_id)

# Output size
concept_name = 'Activity'
concept_name_size = [cat[1] for _, cat in enumerate(dd_all_categories[0]) if cat[0] == concept_name][0]
print("ID concet name in cat list: ", concept_name_size)

# Id of EOS token in activity
eos_value = 'EOS'
eos_id = [v for k, v in dd_all_categories[0][concept_name_id][2].items() if k == eos_value][0]
# Get EOS id of concept name list:
print("ID EOS in concept name tensor: ", eos_id)


[('Activity', 10, {'APPROVED': 1, 'EOS': 2, 'FINAL_APPROVED': 3, 'FOR_APPROVAL': 4, 'Payment Handled': 5, 'REJECTED': 6, 'Request Payment': 7, 'SAVED': 8, 'SUBMITTED': 9}), ('Role', 9, {'ADMINISTRATION': 1, 'BUDGET OWNER': 2, 'EMPLOYEE': 3, 'EOS': 4, 'MISSING': 5, 'PRE_APPROVER': 6, 'SUPERVISOR': 7, 'UNDEFINED': 8})]
[('case_elapsed_time', 1, {})]
domestic_declarations(5) Categorical feature: Activity, Index position in categorical data list: 0
domestic_declarations (5) Total Amount of Category labels: 10
domestic_declarations(5) Categorical feature: Role, Index position in categorical data list: 1
domestic_declarations (5) Total Amount of Category labels: 9


domestic_declarations (5) Numerical feature: case_elapsed_time, Index position in categorical data list: 0
domestic_declarations (5) Amount Numerical: 1
ID concet name in cat list:  0
ID concet name in cat list:  10
ID EOS in concept name tensor:  2


### Input Features for Encoder and Decoder

In [13]:
# Create lists with name of Model features (input)
model_feat_cat = []
model_feat_num = []
for cat in dd_all_categories_cat:
    model_feat_cat.append(cat[0])
for num in dd_all_categories_num:
    model_feat_num.append(num[0])
model_feat = [model_feat_cat, model_feat_num]
print("Input features encoder: ", model_feat)


Input features encoder:  [['Activity', 'Role'], ['case_elapsed_time']]


In [14]:
import sharedCatLSTM.model
importlib.reload(sharedCatLSTM.model)
from sharedCatLSTM.model import SharedCat_LSTM

# Paper hyper-parameters (simple SF / steady TV row).
hidden_size = 50
num_layers = 1
input_size = 1  # sentinel; SharedCat_LSTM computes the real input size from embeddings.

model = SharedCat_LSTM(
    data_set_categories=dd_all_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    model_feat=model_feat,
    input_size=input_size,
    output_size_act=concept_name_size,
)

Data set categories:  ([('Activity', 10, {'APPROVED': 1, 'EOS': 2, 'FINAL_APPROVED': 3, 'FOR_APPROVAL': 4, 'Payment Handled': 5, 'REJECTED': 6, 'Request Payment': 7, 'SAVED': 8, 'SUBMITTED': 9}), ('Role', 9, {'ADMINISTRATION': 1, 'BUDGET OWNER': 2, 'EMPLOYEE': 3, 'EOS': 4, 'MISSING': 5, 'PRE_APPROVER': 6, 'SUPERVISOR': 7, 'UNDEFINED': 8})], [('case_elapsed_time', 1, {})])
Model input features:  [['Activity', 'Role'], ['case_elapsed_time']]
Embeddings:  ModuleList(
  (0): Embedding(10, 6)
  (1): Embedding(9, 5)
)
Total embedding feature size:  11
Number of numerical features:  1
Input feature size (shared LSTM, cat-only):  11
Cells hidden size:  50
Number of LSTM layer:  1


/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [15]:
# SKIPPED — this cell rebinds `model` to FullShared_Join_LSTM, but the config
# (`camargo_model_class="SharedCat_LSTM"`) requires SharedCat. Leaving this
# active produces a checkpoint with shared_lstm (200,10) / lstm_act (200,50)
# that SharedCat_LSTM.load() then rejects with a size-mismatch error.


In [16]:
import training.camargo_ngram_dataset
import training.train_ngram
importlib.reload(training.camargo_ngram_dataset)
importlib.reload(training.train_ngram)
from training.camargo_ngram_dataset import CamargoNGramDataset
from training.train_ngram import NGramTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

# Paper Table 3: n-gram size 5 for short-trace, simple-SF datasets (DomDecl fits this row).
NGRAM_SIZE = 5

ngram_train = CamargoNGramDataset(dd_train_dataset, ngram_size=NGRAM_SIZE,
                                  activity_idx=concept_name_id, eos_idx=eos_id)
ngram_val = CamargoNGramDataset(dd_val_dataset, ngram_size=NGRAM_SIZE,
                                activity_idx=concept_name_id, eos_idx=eos_id)
print(f'n-gram train: {len(ngram_train)} samples (from {len(dd_train_dataset)} base windows)')
print(f'n-gram val:   {len(ngram_val)} samples (from {len(dd_val_dataset)} base windows)')

writer = SummaryWriter(comment="Full_DomesticDeclarations_camargo_sharedcat_role_ngram5")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# lr=1e-3 (paper-aligned Adam default). The earlier 1e-5 was inherited from the
# U-ED-LSTM setup and is too small for the Camargo baseline.
learning_rate = 1e-3
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=0)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4, min_lr=1e-10)

num_epochs = 100
batch_size = 128
shuffle = True

optimize_values = {
    "optimizer": optimizer,
    "scheduler": scheduler,
    "epochs": num_epochs,
    "mini_batches": batch_size,
    "shuffle": shuffle,
}

trainer = NGramTraining(
    model=model,
    device=device,
    data_train=ngram_train,
    data_val=ngram_val,
    optimize_values=optimize_values,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path="../pkl/DomesticDeclarations_camargo_sharedcat_role_ngram5_0515.pkl",
)

trainer.train()

n-gram train: 43531 samples (from 50361 base windows)
n-gram val:   10030 samples (from 11605 base windows)
Device:  cpu
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Scheduler: <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x138617890>
Epochs: 100  Mini-batch: 128  Shuffle: True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], LR: 0.001
Training:   Avg Loss: 0.5099
Validation: Avg Loss: 0.2575
saving model
Epoch [2/100], LR: 0.001
Training:   Avg Loss: 0.2601
Validation: Avg Loss: 0.2475
saving model
Epoch [3/100], LR: 0.001
Training:   Avg Loss: 0.2529
Validation: Avg Loss: 0.2423
saving model
Epoch [4/100], LR: 0.001
Training:   Avg Loss: 0.2511
Validation: Avg Loss: 0.2404
saving model
Epoch [5/100], LR: 0.001
Training:   Avg Loss: 0.2489
Validation: Avg Loss: 0.2412
saving model
Epoch [6/100], LR: 0.001
Training:   Avg Loss: 0.2481
Validation: Avg Loss: 0.2393
saving model
Epoch [7/100], LR: 0.001
Training:   Avg Loss: 0.2481
Validation: Avg Loss: 0.2392
saving model
Epoch [8/100], LR: 0.001
Training:   Avg Loss: 0.2473
Validation: Avg Loss: 0.2391
saving model
Epoch [9/100], LR: 0.001
Training:   Avg Loss: 0.2463
Validation: Avg Loss: 0.2385
saving model
Epoch [10/100], LR: 0.001
Training:   Avg Loss: 0.2461
Validation: Avg Loss: 0.2388
saving model
Epoch [11/100], LR: 0.001
Training:   A